# Tulya Experiment 2 v2 — Ruthless Dual-GPU Falsification

v1 was aborted **before predictor evaluation** because ordinary healthy convergence was incorrectly treated as an early event before the first forecast point.

v2 fixes the event ontology:

- healthy completion = **right-censored / no event**
- actual future regime changes retain event times
- A/B/C feature sets and core kill thresholds are unchanged
- independent runs execute one-per-GPU when Kaggle exposes two GPUs
- diagnostic-only D/in-domain/domain-identity/economic analyses are preregistered and cannot rescue a failed core result

Read the audit files before running:
- `EXPERIMENT2_V1_ABORT.md`
- `EXPERIMENT2_V2_PREREGISTRATION.md`
- `EXPERIMENT2_V2_DIAGNOSTIC_ADDENDUM.md`

Enable **GPU T4 x2** and **Internet** in Kaggle.

In [ ]:
import os, sys, subprocess, json, time
REPO="/kaggle/working/tulya-training-dynamics"
if os.path.exists(REPO):
    subprocess.run(["git","-C",REPO,"pull","--ff-only"],check=True)
else:
    subprocess.run(["git","clone","https://github.com/Vedsaga/tulya-training-dynamics.git",REPO],check=True)
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0,REPO)

import torch, pandas as pd
print("commit:",subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip())
print("torch:",torch.__version__)
print("CUDA available:",torch.cuda.is_available())
print("visible GPUs:",torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}:",torch.cuda.get_device_name(i))

if torch.cuda.device_count()<2:
    print("WARNING: fewer than 2 GPUs visible. The runner still works, but will not get the intended parallel speedup.")

## 1. Frozen audit trail

In [ ]:
print(open("EXPERIMENT2_V1_ABORT.md").read())
print("\n--- V2 PREREGISTRATION ---\n")
print(open("EXPERIMENT2_V2_PREREGISTRATION.md").read())
print("\n--- V2 DIAGNOSTIC ADDENDUM ---\n")
print(open("EXPERIMENT2_V2_DIAGNOSTIC_ADDENDUM.md").read())

## 2. Engineering smoke test

This is not part of the scientific corpus. It checks that v2 event labeling and saving execute.

In [ ]:
from experiment2_core import RunSpec, run_one
SMOKE_ROOT="/kaggle/working/tulya_exp2_v2_smoke"
smoke=run_one(RunSpec("synthetic_sequence_gru","healthy",98765),SMOKE_ROOT)
smoke

## 3. Optional one-seed diversity preflight

This runs 20 jobs (4 domains × 5 configurations × seed 0), using both GPUs.

It is only a run-generation sanity check. **Do not fit or evaluate A/B/C here.**

If a whole domain produces only one outcome class, stop and send the table back before spending time on the remaining seeds. Otherwise continue to the full corpus.

The full cell below is resumable and reuses these 20 v2 runs.

In [ ]:
from experiment2_core import run_suite_parallel

ROOT="/kaggle/working/tulya_exp2_v2"
start=time.time()
preflight=run_suite_parallel(ROOT,seeds=(0,),resume=True)
print("preflight wall time:",time.time()-start)
display(preflight.groupby(["domain","event","event_observed"]).size().rename("runs").reset_index())

preflight_classes=preflight.groupby("domain")["event"].nunique()
display(preflight_classes.rename("outcome_classes"))
if (preflight_classes<2).any():
    print("PRELIMINARY WARNING: at least one domain has only one outcome class on seed 0.")
    print("STOP HERE and send me this table; do not run the blind evaluator.")
else:
    print("Preflight diversity looks plausible. Continue to the full corpus.")

## 4. Full 80-run v2 corpus

Two subprocess workers are created. With Kaggle T4×2, each worker sees exactly one GPU and trains a different run concurrently.

If Kaggle disconnects, rerun this cell. Completed v2 runs are reused.

In [ ]:
start=time.time()
manifest=run_suite_parallel(ROOT,seeds=(0,1,2,3),resume=True)
print("full-suite wall time in this invocation:",time.time()-start)

print("\nObserved event distribution:")
display(manifest.groupby(["domain","event","event_observed"]).size().rename("runs").reset_index())

print("\nInducing configuration -> observed event audit (diagnostic only):")
display(manifest.groupby(["domain","intent","event"]).size().rename("runs").reset_index())

display(manifest)

## 5. Strict data-adequacy gate

No blind evaluation is allowed unless:
- each domain has at least 2 outcome classes;
- at least 3 classes exist globally;
- every class in each held-out domain has at least 2 training-domain runs available.

`no_event` is a legitimate censored outcome class.

In [ ]:
from collections import Counter

per_domain=manifest.groupby("domain")["event"].nunique()
global_classes=manifest["event"].nunique()
problems=[]

if not (per_domain>=2).all():
    problems.append("one or more domains have <2 outcome classes")
if global_classes<3:
    problems.append("fewer than 3 outcome classes globally")

for held in sorted(manifest.domain.unique()):
    te=manifest[manifest.domain==held]
    tr=manifest[manifest.domain!=held]
    for event in sorted(te.event.unique()):
        n=int((tr.event==event).sum())
        if n<2:
            problems.append(f"held-out {held}: class {event!r} has only {n} training-domain runs")

display(manifest.groupby(["domain","event"]).size().rename("runs").reset_index())
if problems:
    print("\nVERDICT: INVALID_DATA_GENERATION")
    for p in problems:
        print(" -",p)
    adequate=False
else:
    print("\nDATA ADEQUACY: PASS — blind evaluation is allowed.")
    adequate=True

## 6. Blind leave-one-domain-out evaluation + preregistered diagnostics

Run only after adequacy passes.

Core systems:

A = learning curves  
B = strong raw telemetry  
C = normalized/canonical dynamics

Diagnostic-only:

D = raw + normalized hybrid

The evaluator also reports:

- whole-run bootstrap AUROC confidence intervals
- within-domain grouped AUROC
- domain/architecture identifiability from B/C/D
- stop-now compute utility including false-stop cost
- an automatic failure-pattern interpretation

**D and the diagnostics cannot rescue a failed C-vs-B core verdict.**

Healthy/no-event runs remain censored risk-set examples. Event-time MAE is computed only for observed events.

In [ ]:
if not adequate:
    raise RuntimeError("Data adequacy failed; blind evaluator intentionally blocked.")

from experiment2_eval import build_table,evaluate

forecast_table=build_table(ROOT)
print("forecast prefixes:",len(forecast_table))
display(forecast_table.groupby(["meta_domain","label_event"]).size().rename("prefixes").reset_index())

folds,result,within_domain,domain_identity=evaluate(ROOT)

print("\nZERO-SHOT LEAVE-ONE-DOMAIN-OUT:")
display(folds)

print("\nWITHIN-DOMAIN GROUPED DIAGNOSTIC:")
display(within_domain)

print("\nDOMAIN/ARCHITECTURE IDENTIFIABILITY DIAGNOSTIC:")
display(domain_identity)

print("\nFAILURE DECOMPOSITION:")
print(json.dumps(result.get("diagnostics",{}),indent=2))

print("\nCORE VERDICT:")
print(json.dumps(result,indent=2))

## 7. Telemetry overhead benchmark

Engineering-only paired run.

In [ ]:
from experiment2_core import benchmark_overhead
overhead=benchmark_overhead(ROOT,domain="synthetic_sequence_gru",seed=991)
overhead

## 8. Final decision

- Core transfer/value gate fails → **KILL_PRODUCT_DIRECTION**
- Core gate passes → candidate earns Experiment 3 on a genuine small LM/SFT workload
- Engineering-only overhead failure can justify one implementation optimization, but not a new feature hypothesis.

In [ ]:
print("BLIND VERDICT:",result["verdict"])
print("OVERHEAD PASS:",overhead["pass_le_0_02"])
print("\nArtifacts:")
for name in [
    "manifest.csv",
    "forecast_table.csv",
    "evaluation_folds.csv",
    "evaluation_summary.json",
    "diagnostic_in_domain.csv",
    "diagnostic_domain_identity.csv",
    "diagnostic_interpretation.json",
]:
    print(os.path.join(ROOT,name))